# Watermelon Cup 2026 Seeder

This notebook automates the setup and seeding of match data for the **Watermelon Cup 2026** tournament in your Firestore database.

## What It Does

1. **Auth**: Initializes the Firebase Admin SDK using your service account key.  
2. **League Lookup/Creation**: Searches for an existing `leagues` document named “Watermelon Cup 2026”. If not found, it creates a new one.  
3. **Match Seeding**: Iterates through a hard‑coded list of all regular‑season matches (week, round, field, home/away teams) and writes each as its own document in the `matches` subcollection under that league. All scores are initialized to `null`.

## How to Use

1. **Upload** your Firebase service account JSON to the Colab runtime (and update the path in the `credentials.Certificate(...)` call).  
2. **Replace** the placeholder `TEAM_X` variables with your actual Firestore team‑document IDs.  
3. **Run** the cells in order:
   - Firebase initialization  
   - League lookup/creation  
   - Match seeding  
4. **Verify** in the Firestore console that:
   - A document exists in `leagues/` with name “Watermelon Cup 2026”.  
   - Under `leagues/{docID}/matches/`, there is one document per match.

> After running, you can build your front‑end to listen to individual match documents and update scores in real time.


In [1]:
import firebase_admin
from firebase_admin import credentials, firestore

In [2]:
# **1. Connect to your Firestore project**
# Replace 'path/to/your/serviceAccountKey.json' with the actual path
cred = credentials.Certificate('C:\\Users\\mdj20\\Downloads\\watermelon-cup-production-firebase-adminsdk-2rmym-f1467bd3a9.json')
firebase_admin.initialize_app(cred)
db = firestore.client()

In [3]:
# ----------------------------
# Get or Create the League Document
# ----------------------------
results = (
    db.collection("leagues")
      .where(field_path="name", op_string="==", value="Watermelon Cup 2026")
      .limit(1)
      .get()
)

if results:
    league_doc_ref = results[0].reference
else:
    league_doc_ref = db.collection("leagues").document()
    league_doc_ref.set({ "name": "Watermelon Cup 2026" })

c:\Users\mdj20\Documents\GitHub\watermelon-cup\.venv\Lib\site-packages\google\cloud\firestore_v1\base_collection.py:317: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)


In [ ]:
# ----------------------------
# Create Team Documents in Firestore
# ----------------------------
# Hardcode name, color, and logo for each of the 6 teams.
# Each team will be created as a document in the "teams" subcollection
# under the league document. The generated doc IDs are stored in TEAM_X variables.

teams_data = [
    {"name": "Italy",     "color": "1428A0", "logo": "https://flagcdn.com/it.svg"},
    {"name": "Brazil",    "color": "FFDF00", "logo": "https://flagcdn.com/br.svg"},
    {"name": "Canada",    "color": "F7F7F5", "logo": "https://flagcdn.com/ca.svg"},
    {"name": "Venezuela", "color": "CF142B", "logo": "https://flagcdn.com/ve.svg"},
    {"name": "Argentina", "color": "000000", "logo": "https://flagcdn.com/ar.svg"},
    {"name": "Guatemala", "color": "009B3A", "logo": "https://flagcdn.com/gt.svg"},
]

teams_ref = league_doc_ref.collection("teams")
team_ids = []

for team in teams_data:
    _, doc_ref = teams_ref.add(team)
    team_ids.append(doc_ref.id)
    print(f"✅ Created team '{team['name']}' with doc ID: {doc_ref.id}")

# Assign to global variables for use in match seeding
TEAM_1, TEAM_2, TEAM_3, TEAM_4, TEAM_5, TEAM_6 = team_ids

print(f"\nTEAM_1 = \"{TEAM_1}\"")
print(f"TEAM_2 = \"{TEAM_2}\"")
print(f"TEAM_3 = \"{TEAM_3}\"")
print(f"TEAM_4 = \"{TEAM_4}\"")
print(f"TEAM_5 = \"{TEAM_5}\"")
print(f"TEAM_6 = \"{TEAM_6}\"")

✅ Created team 'Italy' with doc ID: OiSrxdbbqrCCxJDQZLGJ
✅ Created team 'Brazil' with doc ID: K0leEsnzXhr2O63w2JCN
✅ Created team 'Canada' with doc ID: ngKYazJLzGx0In85Wolg
✅ Created team 'Venezuela' with doc ID: 4PasWD3hmzDZ2NeuIC3c
✅ Created team 'Argentina' with doc ID: pPqBBRTeFjkx9VPG8qRg
✅ Created team 'Guatemala' with doc ID: u67NyrR6JjwcGclVyBpF

TEAM_1 = "OiSrxdbbqrCCxJDQZLGJ"
TEAM_2 = "K0leEsnzXhr2O63w2JCN"
TEAM_3 = "ngKYazJLzGx0In85Wolg"
TEAM_4 = "4PasWD3hmzDZ2NeuIC3c"
TEAM_5 = "pPqBBRTeFjkx9VPG8qRg"
TEAM_6 = "u67NyrR6JjwcGclVyBpF"


In [5]:
# ----------------------------
# List Team Document IDs and Names
# ----------------------------

# Assumes league_doc_ref is already defined by the previous cell
teams_ref = league_doc_ref.collection("teams")
teams = teams_ref.stream()

print("📋 Teams in Watermelon Cup 2026:")
for team in teams:
    data = team.to_dict()
    team_name = data.get("name", "<no name field>")
    print(f"• Team Doc ID: {team.id}, Team Name: {team_name}")

📋 Teams in Watermelon Cup 2026:
• Team Doc ID: 4PasWD3hmzDZ2NeuIC3c, Team Name: Venezuela
• Team Doc ID: K0leEsnzXhr2O63w2JCN, Team Name: Brazil
• Team Doc ID: OiSrxdbbqrCCxJDQZLGJ, Team Name: Italy
• Team Doc ID: ngKYazJLzGx0In85Wolg, Team Name: Canada
• Team Doc ID: pPqBBRTeFjkx9VPG8qRg, Team Name: Argentina
• Team Doc ID: u67NyrR6JjwcGclVyBpF, Team Name: Guatemala


In [6]:
# ----------------------------
# Hardcoded Match Data for the Season
# ----------------------------
matches = [
    # Week 1, Round 1
    {"homeTeamId": TEAM_4, "awayTeamId": TEAM_5, "homeScore": None, "awayScore": None, "fieldLocation": "Wakeman 1", "week": 1, "round": 1},
    {"homeTeamId": TEAM_3, "awayTeamId": TEAM_6, "homeScore": None, "awayScore": None, "fieldLocation": "Wakeman 2", "week": 1, "round": 1},
    {"homeTeamId": TEAM_1, "awayTeamId": TEAM_2, "homeScore": None, "awayScore": None, "fieldLocation": "Grass",     "week": 1, "round": 1},
    # Week 1, Round 2
    {"homeTeamId": TEAM_5, "awayTeamId": TEAM_1, "homeScore": None, "awayScore": None, "fieldLocation": "Wakeman 1", "week": 1, "round": 2},
    {"homeTeamId": TEAM_2, "awayTeamId": TEAM_6, "homeScore": None, "awayScore": None, "fieldLocation": "Wakeman 2", "week": 1, "round": 2},
    {"homeTeamId": TEAM_3, "awayTeamId": TEAM_4, "homeScore": None, "awayScore": None, "fieldLocation": "Grass",     "week": 1, "round": 2},

    # Week 2, Round 1
    {"homeTeamId": TEAM_2, "awayTeamId": TEAM_3, "homeScore": None, "awayScore": None, "fieldLocation": "Wakeman 1", "week": 2, "round": 1},
    {"homeTeamId": TEAM_1, "awayTeamId": TEAM_4, "homeScore": None, "awayScore": None, "fieldLocation": "Wakeman 2", "week": 2, "round": 1},
    {"homeTeamId": TEAM_5, "awayTeamId": TEAM_6, "homeScore": None, "awayScore": None, "fieldLocation": "Grass",     "week": 2, "round": 1},
    # Week 2, Round 2
    {"homeTeamId": TEAM_2, "awayTeamId": TEAM_5, "homeScore": None, "awayScore": None, "fieldLocation": "Wakeman 1", "week": 2, "round": 2},
    {"homeTeamId": TEAM_4, "awayTeamId": TEAM_6, "homeScore": None, "awayScore": None, "fieldLocation": "Wakeman 2", "week": 2, "round": 2},
    {"homeTeamId": TEAM_1, "awayTeamId": TEAM_3, "homeScore": None, "awayScore": None, "fieldLocation": "Grass",     "week": 2, "round": 2},

    # Week 3, Round 1
    {"homeTeamId": TEAM_1, "awayTeamId": TEAM_4, "homeScore": None, "awayScore": None, "fieldLocation": "Wakeman 1", "week": 3, "round": 1},
    {"homeTeamId": TEAM_3, "awayTeamId": TEAM_6, "homeScore": None, "awayScore": None, "fieldLocation": "Wakeman 2", "week": 3, "round": 1},
    {"homeTeamId": TEAM_2, "awayTeamId": TEAM_5, "homeScore": None, "awayScore": None, "fieldLocation": "Grass",     "week": 3, "round": 1},
    # Week 3, Round 2
    {"homeTeamId": TEAM_5, "awayTeamId": TEAM_3, "homeScore": None, "awayScore": None, "fieldLocation": "Wakeman 1", "week": 3, "round": 2},
    {"homeTeamId": TEAM_1, "awayTeamId": TEAM_2, "homeScore": None, "awayScore": None, "fieldLocation": "Wakeman 2", "week": 3, "round": 2},
    {"homeTeamId": TEAM_4, "awayTeamId": TEAM_6, "homeScore": None, "awayScore": None, "fieldLocation": "Grass",     "week": 3, "round": 2},

    # Week 4, Round 1
    {"homeTeamId": TEAM_4, "awayTeamId": TEAM_5, "homeScore": None, "awayScore": None, "fieldLocation": "Wakeman 1", "week": 4, "round": 1},
    {"homeTeamId": TEAM_2, "awayTeamId": TEAM_3, "homeScore": None, "awayScore": None, "fieldLocation": "Wakeman 2", "week": 4, "round": 1},
    {"homeTeamId": TEAM_1, "awayTeamId": TEAM_6, "homeScore": None, "awayScore": None, "fieldLocation": "Grass",     "week": 4, "round": 1},
    # Week 4, Round 2
    {"homeTeamId": TEAM_1, "awayTeamId": TEAM_3, "homeScore": None, "awayScore": None, "fieldLocation": "Wakeman 1", "week": 4, "round": 2},
    {"homeTeamId": TEAM_5, "awayTeamId": TEAM_6, "homeScore": None, "awayScore": None, "fieldLocation": "Wakeman 2", "week": 4, "round": 2},
    {"homeTeamId": TEAM_2, "awayTeamId": TEAM_4, "homeScore": None, "awayScore": None, "fieldLocation": "Grass",     "week": 4, "round": 2},
]

In [7]:
# ----------------------------
# Reference (and thus create) the "matches" subcollection
# ----------------------------
matches_ref = league_doc_ref.collection("matches")

In [8]:
# ----------------------------
# Seed Matches into Firestore
# ----------------------------
for match in matches:
    matches_ref.add(match)

In [9]:
# ----------------------------
# Fetch Team Names and Display Aligned Matches (fixed sort)
# ----------------------------

# 1. Load all teams into a lookup dict
teams = list(league_doc_ref.collection("teams").stream())
team_names = {t.id: t.to_dict().get("name", "<no name>") for t in teams}

# 2. Fetch all matches
matches = list(league_doc_ref.collection("matches").stream())

# 3. Build rows of formatted strings
rows = []
for m in matches:
    d = m.to_dict()
    id_str    = f"Doc ID: {m.id}"
    match_str = f"Week {d['week']}, Round {d['round']}, Field {d['fieldLocation']}"
    vs_str    = f"{team_names.get(d['homeTeamId'], '')} vs {team_names.get(d['awayTeamId'], '')}"
    score_str = f"Score: {d['homeScore']}–{d['awayScore']}"
    rows.append((id_str, match_str, vs_str, score_str))

# 4. Determine padding widths for first three columns
col_widths = [max(len(r[i]) for r in rows) for i in range(3)]

# 5. Sort rows by week & round
def sort_key(row):
    parts = [p.strip() for p in row[1].split(',')]
    week_num  = int(parts[0].split()[1])    # "Week X"
    round_num = int(parts[1].split()[1])    # "Round Y"
    return (week_num, round_num)

rows.sort(key=sort_key)

# 6. Print aligned output
print("📋 Matches with Team Names:")
for id_str, match_str, vs_str, score_str in rows:
    print(
        f"• {id_str.ljust(col_widths[0])} | "
        f"{match_str.ljust(col_widths[1])} | "
        f"{vs_str.ljust(col_widths[2])} | "
        f"{score_str}"
    )

📋 Matches with Team Names:
• Doc ID: 0pfGb12nW2sSj98iMcKB | Week 1, Round 1, Field Wakeman 1 | Venezuela vs Argentina | Score: None–None
• Doc ID: QzwAOvT6qGDATg8eG4YM | Week 1, Round 1, Field Wakeman 2 | Canada vs Guatemala    | Score: None–None
• Doc ID: TOHudi3eIR5JpMTA5TEP | Week 1, Round 1, Field Grass     | Italy vs Brazil        | Score: None–None
• Doc ID: Gp3UbCcLRGVtIcTjmbe9 | Week 1, Round 2, Field Wakeman 2 | Brazil vs Guatemala    | Score: None–None
• Doc ID: bMEI51WZCxdEgZgcGO8Y | Week 1, Round 2, Field Grass     | Canada vs Venezuela    | Score: None–None
• Doc ID: l11EgPfcGqUntYxSrUgo | Week 1, Round 2, Field Wakeman 1 | Argentina vs Italy     | Score: None–None
• Doc ID: 3GY3452ZlkME7aG1vVls | Week 2, Round 1, Field Wakeman 1 | Brazil vs Canada       | Score: None–None
• Doc ID: CkEOg7nr0jnyt9oMdeP4 | Week 2, Round 1, Field Grass     | Argentina vs Guatemala | Score: None–None
• Doc ID: Hf4vvNIHInsD6pdVbFci | Week 2, Round 1, Field Wakeman 2 | Italy vs Venezuela     | 